# Convolution & Image Filtering

Filtering companion for *ML 03 · Convolution, Image Filtering and CNNs*. This notebook covers kernel experiments; the merged article also includes a standalone CNN training example. Runs on Google Colab with no setup.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from skimage import data  # sample images; preinstalled on Colab

img = data.camera().astype(np.float32) / 255  # (512, 512) grayscale
plt.imshow(img, cmap="gray"); plt.axis("off");

## From scratch with NumPy

In [ ]:
def conv2d(img, K):
    """Cross-correlation of a (H, W) image with a (3, 3) kernel, edge-replicated borders."""
    H, W = img.shape
    p = np.pad(img, 1, mode="edge")
    out = np.zeros_like(img)
    for i in range(3):
        for j in range(3):
            out += K[i, j] * p[i:i + H, j:j + W]
    return out

kernels = {
    "box blur": np.ones((3, 3), np.float32) / 9,
    "sharpen": np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], np.float32),
    "sobel x": np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], np.float32),
    "laplacian": np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], np.float32),
}

fig, axes = plt.subplots(1, len(kernels), figsize=(16, 4))
for ax, (name, K) in zip(axes, kernels.items()):
    out = conv2d(img, K)
    ax.imshow(out, cmap="gray"); ax.set_title(name); ax.axis("off")

## Check against OpenCV

In [ ]:
K = kernels["sobel x"]
ours = conv2d(img, K)
theirs = cv2.filter2D(img, -1, K, borderType=cv2.BORDER_REPLICATE)
print("max abs difference:", np.abs(ours - theirs).max())

## Exercises

1. Design a kernel that detects only horizontal edges.
2. Why must a blur kernel's weights sum to 1? Try a kernel that sums to 2.
3. Show that applying a 3×3 box blur twice equals one 5×5 kernel.